In [ ]:
import json
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re

In [ ]:
# --------------------------------------------------------
# CONFIGURACIÓN
# --------------------------------------------------------

URL_INDICE = "https://www.upv.es/entidades/edoctorado/todos-los-programas-de-doctorado-ofertados-en-upv/"
BASE = "https://www.upv.es"

SALIDA = "doctorados.json"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}


In [ ]:
# --------------------------------------------------------
# DESCARGA
# --------------------------------------------------------

r = requests.get(
    URL_INDICE,
    headers=HEADERS,
    timeout=30
)

r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")



In [ ]:
# --------------------------------------------------------
# EXTRACCIÓN
# --------------------------------------------------------

vistos = set()
doctorados = []


def parece_programa(url, texto):

    cadena = (url + " " + texto).lower()

    patrones_validos = [
        "programa",
        "doctorado",
        "pd"
    ]

    patrones_excluir = [
        "acceso",
        "admision",
        "matricula",
        "normativa",
        "tesis",
        "calendario",
        "contacto",
        "noticias"
    ]

    if any(x in cadena for x in patrones_excluir):
        return False

    if not any(x in cadena for x in patrones_validos):
        return False

    return True



# buscamos enlaces dentro del contenido principal
# for a in soup.find_all("a", href=True):

#     href = a["href"].strip()
#     nombre = a.get_text(" ", strip=True)

#     if not nombre:
#         continue


#     url = urljoin(BASE, href)


#     if url in vistos:
#         continue


#     if not url.startswith(BASE):
#         continue


#     if not parece_programa(url, nombre):
#         continue


#     vistos.add(url)


#     doctorados.append({

#         "acro": "",

#         "tipo": [
#             "D"
#         ],

#         "nom": nombre,

#         "url": url,

#         "centros": [],

#         "ramas": [],

#         "mostrar_leye": False,

#         "id_leye": "VALIDO",

#         "mostrar_sello": False,

#         "id_sello": None,

#         "mostrar_pac": False,

#         "pac": None,

#         "mostrar_masInfo": False,

#         "id_info": None

#     })

for a in soup.find_all("a", href=True):

    nombre = a.get_text(" ", strip=True)
    href = a["href"]

    # SOLO programas reales
    if not nombre.startswith("Programa de"):
        continue

    url = urljoin(BASE, href)

    if url in vistos:
        continue

    vistos.add(url)

    doctorados.append({
        "acro": "",
        "tipo": ["D"],
        "nom": nombre,
        "url": url,
        "centros": [],
        "ramas": [],
        "mostrar_leye": False,
        "id_leye": "VALIDO",
        "mostrar_sello": False,
        "id_sello": None,
        "mostrar_pac": False,
        "pac": None,
        "mostrar_masInfo": False,
        "id_info": None
    })


In [ ]:
# --------------------------------------------------------
# GUARDADO
# --------------------------------------------------------

datos = {
    "titulaciones": doctorados
}


with open(
    SALIDA,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        datos,
        f,
        ensure_ascii=False,
        indent=2
    )


print("Doctorados encontrados:", len(doctorados))
print("Archivo creado:", SALIDA)

Doctorados encontrados: 32
Archivo creado: doctorados.json


In [ ]:
import json

d = json.load(open("doctorados.json"))

for x in d["titulaciones"][:10]:
    print(x["nom"])
    print(x["url"])
    print()

Programa de doctorado en Recursos y Tecnologías Agrícolas
https://www.upv.es/entidades/edoctorado/programa-de-doctorado-en-recursos-y-tecnologias-agricolas/

Programa de doctorado en Ciencia y Tecnología de la Producción Animal
https://www.upv.es/entidades/edoctorado/programa-de-doctorado-en-ciencia-y-tecnologia-de-la-produccion-animal/

Programa de doctorado en Ciencia, Tecnología y Gestión Alimentaria
https://www.upv.es/entidades/edoctorado/programa-de-doctorado-en-ciencia-tecnologia-y-gestion-alimentaria/

Programa de doctorado en Biotecnología
https://www.upv.es/entidades/edoctorado/programa-de-doctorado-en-biotecnologia/

Programa de doctorado en Arquitectura, Edificación, Urbanística y Paisaje
https://www.upv.es/entidades/edoctorado/programa-de-doctorado-en-arquitectura-edificacion-urbanistica-y-paisaje/

Programa de doctorado en Arquitectura, Edificación, Patrimonio y Ciudad (*)
https://www.upv.es/entidades/edoctorado/programa-de-doctorado-en-arquitectura-edificacion-patrimonio-

In [ ]:
print(len(doctorados))

for d in doctorados:
    print(d["nom"], " -> ", d["url"])

32
Programa de doctorado en Recursos y Tecnologías Agrícolas  ->  https://www.upv.es/entidades/edoctorado/programa-de-doctorado-en-recursos-y-tecnologias-agricolas/
Programa de doctorado en Ciencia y Tecnología de la Producción Animal  ->  https://www.upv.es/entidades/edoctorado/programa-de-doctorado-en-ciencia-y-tecnologia-de-la-produccion-animal/
Programa de doctorado en Ciencia, Tecnología y Gestión Alimentaria  ->  https://www.upv.es/entidades/edoctorado/programa-de-doctorado-en-ciencia-tecnologia-y-gestion-alimentaria/
Programa de doctorado en Biotecnología  ->  https://www.upv.es/entidades/edoctorado/programa-de-doctorado-en-biotecnologia/
Programa de doctorado en Arquitectura, Edificación, Urbanística y Paisaje  ->  https://www.upv.es/entidades/edoctorado/programa-de-doctorado-en-arquitectura-edificacion-urbanistica-y-paisaje/
Programa de doctorado en Arquitectura, Edificación, Patrimonio y Ciudad (*)  ->  https://www.upv.es/entidades/edoctorado/programa-de-doctorado-en-arquitec

In [1]:
from google.colab import drive
import os
import json

# Montar Drive
drive.mount('/content/drive')


# Buscar la carpeta donde está el notebook/programa actual
nombre_programa = "SacarJSON_Doctorado.ipynb"   # cambia esto si tu archivo se llama distinto

ruta_programa = None

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if nombre_programa in files:
        ruta_programa = root
        break


if ruta_programa is None:
    raise Exception("No se ha encontrado la carpeta del programa")


# Crear carpeta JSONs al lado del programa
carpeta_json = os.path.join(ruta_programa, "JSONs")

os.makedirs(carpeta_json, exist_ok=True)


# Guardar JSON
ruta_json = os.path.join(carpeta_json, "doctorados_upv.json")

with open(
    ruta_json,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        datos,
        f,
        ensure_ascii=False,
        indent=2
    )


print("Guardado en:")
print(ruta_json)

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# ADAPTAR JSON DOCTORADO A FORMATO MASTERES PARA PROGRAMA EXTRAE MASTERES
import json
import re
import os

# Ruta del JSON original que generaste antes
JSON_ORIGINAL = "doctorados.json"

# Nombre del nuevo JSON compatible
JSON_ADAPTADO = "doctorados_upv.json"


# Cargar JSON original
with open(JSON_ORIGINAL, "r", encoding="utf-8") as f:
    datos = json.load(f)


titulaciones_nuevas = []


for d in datos["titulaciones"]:

    nombre = d["nom"]

    # Crear acrónimo único para el estado del extractor
    acro = re.sub(
        r'[^\w\-]',
        '_',
        nombre
    )

    # Mantener URL relativa tipo UPV
    url = d["url"]

    if url.startswith("https://www.upv.es"):
        url = url.replace("https://www.upv.es", "")


    titulaciones_nuevas.append({

        "acro": acro,

        "tipo": [
            "D"
        ],

        "nom": nombre,

        "url": url,

        "centros": d.get("centros", []),

        "ramas": d.get("ramas", []),

        "mostrar_leye": False,

        "id_leye": "VALIDO",

        "mostrar_sello": False,

        "id_sello": None,

        "mostrar_pac": False,

        "pac": None,

        "mostrar_masInfo": False,

        "id_info": None
    })


# Crear estructura compatible con masteres

datos_adaptados = {

    "titulaciones": titulaciones_nuevas,

    "ramas": [],

    "campus": [],

    "centros": [],

    "tipos": [
        {
            "tipo": "D",
            "texto": "Doctorado"
        }
    ]
}


with open(
    JSON_ADAPTADO,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        datos_adaptados,
        f,
        ensure_ascii=False,
        indent=2
    )


print(
    f"Creado {JSON_ADAPTADO}: {len(titulaciones_nuevas)} doctorados"
)

Creado doctorados_upv.json: 32 doctorados


In [ ]:
import json

with open(
    "doctorados_upv.json",
    encoding="utf-8"
) as f:
    datos = json.load(f)


print("Campos superiores:")
print(datos.keys())

print()
print("Número de doctorados:")
print(len(datos["titulaciones"]))

print()
print("Primeros elementos:")

for d in datos["titulaciones"][:5]:

    print("----------------")
    print("acro:", d["acro"][:50])
    print("nom :", d["nom"])
    print("url :", d["url"])
    print("tipo:", d["tipo"])

Campos superiores:
dict_keys(['titulaciones', 'ramas', 'campus', 'centros', 'tipos'])

Número de doctorados:
32

Primeros elementos:
----------------
acro: Programa_de_doctorado_en_Recursos_y_Tecnologías_Ag
nom : Programa de doctorado en Recursos y Tecnologías Agrícolas
url : /entidades/edoctorado/programa-de-doctorado-en-recursos-y-tecnologias-agricolas/
tipo: ['D']
----------------
acro: Programa_de_doctorado_en_Ciencia_y_Tecnología_de_l
nom : Programa de doctorado en Ciencia y Tecnología de la Producción Animal
url : /entidades/edoctorado/programa-de-doctorado-en-ciencia-y-tecnologia-de-la-produccion-animal/
tipo: ['D']
----------------
acro: Programa_de_doctorado_en_Ciencia__Tecnología_y_Ges
nom : Programa de doctorado en Ciencia, Tecnología y Gestión Alimentaria
url : /entidades/edoctorado/programa-de-doctorado-en-ciencia-tecnologia-y-gestion-alimentaria/
tipo: ['D']
----------------
acro: Programa_de_doctorado_en_Biotecnología
nom : Programa de doctorado en Biotecnología
url : /e

In [ ]:
import os
import json

# carpeta donde estás trabajando
CARPETA = os.getcwd()

CARPETA_JSON = os.path.join(CARPETA, "JSONs")

os.makedirs(
    CARPETA_JSON,
    exist_ok=True
)


RUTA = os.path.join(
    CARPETA_JSON,
    "doctorados_upv.json"
)


with open(
    RUTA,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        datos_adaptados,
        f,
        ensure_ascii=False,
        indent=2
    )


print("Guardado en:")
print(RUTA)

Guardado en:
/content/JSONs/doctorados_upv.json
